In [2]:
# If needed (only once per env):
# !pip install pandas numpy scikit-learn matplotlib seaborn torch --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [5]:
DATA_PATH = r"C:\Users\admin\Desktop\Forex_algo\code\Data\EUR_USD_H1.parquet"

df = pd.read_parquet(DATA_PATH)

df.head(), df.tail(), (df.time.min(), df.time.max(), df.shape)


(                       time  volume    mid_o    mid_h    mid_l    mid_c  \
 0 2016-01-07 00:00:00+00:00     542  1.07764  1.07832  1.07744  1.07778   
 1 2016-01-07 01:00:00+00:00    3167  1.07776  1.08100  1.07748  1.08029   
 2 2016-01-07 02:00:00+00:00    1567  1.08026  1.08176  1.07996  1.08152   
 3 2016-01-07 03:00:00+00:00     914  1.08156  1.08257  1.08150  1.08187   
 4 2016-01-07 04:00:00+00:00     649  1.08190  1.08256  1.08156  1.08236   
 
      bid_o    bid_h    bid_l    bid_c    ask_o    ask_h    ask_l    ask_c  
 0  1.07757  1.07823  1.07735  1.07770  1.07772  1.07840  1.07752  1.07787  
 1  1.07768  1.08092  1.07740  1.08020  1.07784  1.08109  1.07756  1.08038  
 2  1.08018  1.08169  1.07987  1.08144  1.08035  1.08184  1.08005  1.08159  
 3  1.08147  1.08249  1.08142  1.08178  1.08164  1.08265  1.08157  1.08196  
 4  1.08182  1.08247  1.08147  1.08228  1.08199  1.08264  1.08163  1.08245  ,
                            time  volume    mid_o    mid_h    mid_l    mid_c  \

In [6]:
df_feat = df.copy()

# Ensure sorted by time
df_feat = df_feat.sort_values("time").reset_index(drop=True)

# Base price (we'll use mid close)
price = df_feat["mid_c"]

# Basic returns
df_feat["ret_1"]  = price.pct_change(1)
df_feat["ret_3"]  = price.pct_change(3)
df_feat["ret_6"]  = price.pct_change(6)
df_feat["ret_12"] = price.pct_change(12)

# Rolling volatility and volume
df_feat["vol_10"] = df_feat["volume"].rolling(10).mean()
df_feat["vol_20"] = df_feat["volume"].rolling(20).mean()

# Simple moving averages
df_feat["ma_10"] = price.rolling(10).mean()
df_feat["ma_20"] = price.rolling(20).mean()
df_feat["dist_ma10"] = (price - df_feat["ma_10"]) / df_feat["ma_10"]
df_feat["dist_ma20"] = (price - df_feat["ma_20"]) / df_feat["ma_20"]

# --- Target: future direction over next N bars ---
horizon = 6  # 6 hours
future_price = price.shift(-horizon)
future_ret = (future_price - price) / price

df_feat["future_ret"] = future_ret
df_feat["target"] = (df_feat["future_ret"] > 0).astype(int)

# Drop last `horizon` rows that have NaN future_ret
df_feat = df_feat.iloc[:-horizon].copy()

df_feat[["time", "mid_c", "future_ret", "target"]].tail(10)


,time,mid_c,future_ret,target
61457,2025-11-21 06:00:00+00:00,1.15414,-0.001447,0
61458,2025-11-21 07:00:00+00:00,1.15414,-0.001785,0
61459,2025-11-21 08:00:00+00:00,1.15410,-0.002339,0
61460,2025-11-21 09:00:00+00:00,1.15252,-0.001926,0
61461,2025-11-21 10:00:00+00:00,1.15200,-0.001302,0
61462,2025-11-21 11:00:00+00:00,1.15060,0.000104,1
61463,2025-11-21 12:00:00+00:00,1.15247,-0.001779,0
61464,2025-11-21 13:00:00+00:00,1.15208,-0.000651,0
61465,2025-11-21 14:00:00+00:00,1.15140,0.000313,1
61466,2025-11-21 15:00:00+00:00,1.15030,0.000869,1


In [7]:
df_feat["target"].value_counts(normalize=True)


target
1    0.503441
0    0.496559
Name: proportion, dtype: float64

In [8]:
feature_cols = [
    "mid_o", "mid_h", "mid_l", "mid_c",
    "volume",
    "ret_1", "ret_3", "ret_6", "ret_12",
    "vol_10", "vol_20",
    "ma_10", "ma_20", "dist_ma10", "dist_ma20"
]

# Drop rows with NaN from rolling features
df_feat = df_feat.dropna(subset=feature_cols + ["target"]).reset_index(drop=True)

X_all = df_feat[feature_cols].values.astype(np.float32)
y_all = df_feat["target"].values.astype(np.int64)

X_all.shape, y_all.shape


((61448, 15), (61448,))

In [9]:
n = len(df_feat)
train_end = int(n * 0.7)
val_end   = int(n * 0.85)

X_train_raw = X_all[:train_end]
y_train     = y_all[:train_end]

X_val_raw   = X_all[train_end:val_end]
y_val       = y_all[train_end:val_end]

X_test_raw  = X_all[val_end:]
y_test      = y_all[val_end:]

X_train_raw.shape, X_val_raw.shape, X_test_raw.shape


((43013, 15), (9217, 15), (9218, 15))

In [10]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

X_train[:3]


array([[-0.74182206, -0.695027  , -0.7426367 , -0.68893266, -0.14074461,
         2.6944153 ,  3.824079  ,  2.7687924 ,  4.086545  , -0.1666802 ,
        -0.29930684, -0.8075881 , -0.85523903,  3.6924942 ,  3.5848522 ],
       [-0.68838793, -0.6864376 , -0.6880092 , -0.68249893, -0.08560527,
         0.3316461 ,  3.8660429 ,  3.4750774 ,  3.513302  , -0.1517667 ,
        -0.25559896, -0.79021233, -0.8412188 ,  3.3482087 ,  3.4183702 ],
       [-0.68249106, -0.6871543 , -0.6810459 , -0.68928844, -0.65146154,
        -0.3491465 ,  1.5519166 ,  3.9647837 ,  2.3379412 , -0.213146  ,
        -0.3133952 , -0.7787336 , -0.82978046,  2.7771444 ,  3.0225127 ]],
      dtype=float32)

In [11]:
window_size = 32


In [12]:
def make_windows(X, y, window_size):
    """
    X: (N, F)
    y: (N,)
    Returns:
      X_win: (N_win, window_size, F)
      y_win: (N_win,)
    """
    X_list = []
    y_list = []
    for i in range(window_size, len(X)):
        X_list.append(X[i-window_size:i])
        y_list.append(y[i])
    return np.stack(X_list), np.array(y_list)

window_size = 32

X_train_win, y_train_win = make_windows(X_train, y_train, window_size)
X_val_win,   y_val_win   = make_windows(X_val,   y_val,   window_size)
X_test_win,  y_test_win  = make_windows(X_test,  y_test,  window_size)

X_train_win.shape, y_train_win.shape


((42981, 32, 15), (42981,))

In [13]:
class ForexCNNDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, T, F)
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # (T, F) -> (F, T)
        x = self.X[idx].permute(1, 0)   # channels = features
        y = self.y[idx]
        return x, y

train_ds = ForexCNNDataset(X_train_win, y_train_win)
val_ds   = ForexCNNDataset(X_val_win,   y_val_win)
test_ds  = ForexCNNDataset(X_test_win,  y_test_win)

batch_size = 256

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, drop_last=False)

len(train_ds), len(val_ds), len(test_ds)


(42981, 9185, 9186)

In [14]:
class ForexCNN(nn.Module):
    def __init__(self, n_features, window_size, n_classes=2):
        super().__init__()
        
        # Input: (batch, n_features, window_size)
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(64)
        
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # After two pools, seq_len reduced from window_size -> window_size / 4 (roughly)
        seq_out = window_size // 4
        
        self.fc1 = nn.Linear(64 * seq_out, 64)
        self.fc2 = nn.Linear(64, n_classes)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # x: (batch, n_features, window_size)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.pool(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

n_features = X_train_win.shape[2]

model = ForexCNN(n_features=n_features, window_size=window_size, n_classes=2).to(device)
model


ForexCNN(
  (conv1): Conv1d(15, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=512, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=2, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (relu): ReLU()
)

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * X_batch.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == y_batch).sum().item()
        total_samples += X_batch.size(0)
    
    return total_loss / total_samples, total_correct / total_samples


def eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            preds = outputs.argmax(dim=1)
            total_correct += (preds == y_batch).sum().item()
            total_samples += X_batch.size(0)
            
            all_preds.append(preds.cpu().numpy())
            all_targets.append(y_batch.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    return total_loss / total_samples, total_correct / total_samples, all_preds, all_targets



In [16]:
n_epochs = 20

best_val_acc = 0.0
best_state = None

for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = eval_model(model, val_loader, criterion, device)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()
    
    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

# Load best validation weights
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model with val acc = {best_val_acc:.4f}")


Epoch 01 | Train Loss: 0.6943 Acc: 0.5088 | Val Loss: 0.6922 Acc: 0.5180
Epoch 02 | Train Loss: 0.6914 Acc: 0.5202 | Val Loss: 0.6924 Acc: 0.5161
Epoch 03 | Train Loss: 0.6903 Acc: 0.5248 | Val Loss: 0.6958 Acc: 0.5022
Epoch 04 | Train Loss: 0.6896 Acc: 0.5258 | Val Loss: 0.6957 Acc: 0.5092
Epoch 05 | Train Loss: 0.6882 Acc: 0.5336 | Val Loss: 0.6997 Acc: 0.5068
Epoch 06 | Train Loss: 0.6858 Acc: 0.5425 | Val Loss: 0.6960 Acc: 0.5079
Epoch 07 | Train Loss: 0.6850 Acc: 0.5415 | Val Loss: 0.6966 Acc: 0.5033
Epoch 08 | Train Loss: 0.6819 Acc: 0.5505 | Val Loss: 0.7034 Acc: 0.4885
Epoch 09 | Train Loss: 0.6800 Acc: 0.5551 | Val Loss: 0.7009 Acc: 0.5046
Epoch 10 | Train Loss: 0.6756 Acc: 0.5634 | Val Loss: 0.7105 Acc: 0.4876
Epoch 11 | Train Loss: 0.6722 Acc: 0.5710 | Val Loss: 0.7064 Acc: 0.4998
Epoch 12 | Train Loss: 0.6668 Acc: 0.5816 | Val Loss: 0.7256 Acc: 0.4960
Epoch 13 | Train Loss: 0.6635 Acc: 0.5841 | Val Loss: 0.7183 Acc: 0.5069
Epoch 14 | Train Loss: 0.6590 Acc: 0.5930 | Val Los

In [17]:
test_loss, test_acc, y_pred_test, y_true_test = eval_model(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_true_test, y_pred_test, digits=4))

print("\nConfusion matrix:")
print(confusion_matrix(y_true_test, y_pred_test))


Test Loss: 0.8194, Test Acc: 0.5017

Classification report:
              precision    recall  f1-score   support

           0     0.4918    0.6184    0.5479      4484
           1     0.5176    0.3905    0.4451      4702

    accuracy                         0.5017      9186
   macro avg     0.5047    0.5044    0.4965      9186
weighted avg     0.5050    0.5017    0.4953      9186


Confusion matrix:
[[2773 1711]
 [2866 1836]]
